# 02. Feature Engineering
## New Zealand Electricity Demand Forecasting

### 1. Introduction

This notebook focuses on preparing historical New Zealand electricity demand data for machine learning.

The dataset contains electricity consumption recorded at 30-minute intervals across five New Zealand electricity regions, covering November 2024 to October 2026.

### Objective

Develop features that help machine learning models predict regional electricity demand 30 minutes ahead.

### Target Variable

**Demand (GWh)** — electricity consumed within a 30-minute trading period.

### Feature Engineering Plan

1. Load and validate the dataset.
2. Create calendar-based features.
3. Generate historical demand (lag) features.
4. Calculate rolling averages.
5. Prepare chronological training and testing datasets.

### Modelling Considerations

- Preserve chronological order.
- Avoid data leakage from future observations.
- Account for missing trading periods.
- Consider daylight saving time when creating historical demand features.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Locate the data folder
data_folder = Path.cwd().parent / "data"

# Find available CSV files
csv_files = list(data_folder.glob("*.csv"))

print("Available datasets:")

for file in csv_files:
    print(file.name)

Available datasets:
Demand_trends_20261118145637.csv


## 2. Loading the Cleaned Dataset

The cleaned electricity demand dataset was exported from the exploratory data analysis notebook (`01_exploratory_data_analysis.ipynb`).

The dataset contains 174,936 observations across five New Zealand electricity regions, covering November 2024 to October 2026.

### Variables

- **Period start:** Beginning of the trading period.
- **Period end:** End of the trading period.
- **Region ID:** Unique regional identifier.
- **Region:** Name of the electricity region.
- **Demand (GWh):** Electricity consumed during each 30-minute trading period.

### Data Quality

The dataset has already undergone initial cleaning, including the removal of observations with missing electricity demand values.

The cleaned dataset is loaded directly to ensure consistency and reproducibility across notebooks.

In [3]:
# Load the cleaned dataset
clean_path = data_folder / "electricity_demand_clean.csv"

df = pd.read_csv(
    clean_path,
    parse_dates=["Period start", "Period end"]
)

# Display dataset information
print("Dataset shape:", df.shape)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

display(df.head())

Dataset shape: (174936, 5)

Data types:
Period start    datetime64[us]
Period end      datetime64[us]
Region ID                  str
Region                     str
Demand (GWh)           float64
dtype: object

Missing values:
Period start    0
Period end      0
Region ID       0
Region          0
Demand (GWh)    0
dtype: int64


,Period start,Period end,Region ID,Region,Demand (GWh)
0,2024-11-01,2024-11-01 00:30:00,UNI,Upper North Island,0.541
1,2024-11-01,2024-11-01 00:30:00,CNI,Central North Island,0.377
2,2024-11-01,2024-11-01 00:30:00,LNI,Lower North Island,0.246
3,2024-11-01,2024-11-01 00:30:00,USI,Upper South Island,0.275
4,2024-11-01,2024-11-01 00:30:00,LSI,Lower South Island,0.433


## 3. Feature Engineering

### 3.1 Calendar-Based Features

Electricity demand varies according to the time of day, day of the week, and season.

The exploratory data analysis identified higher electricity consumption during winter, increased demand during morning and evening hours, and differences between weekdays and weekends.

To capture these patterns, four calendar-based features will be created:

- **Hour:** Hour of the day (0–23).
- **Day_of_week:** Day of the week, where Monday = 0 and Sunday = 6.
- **Month:** Calendar month (1–12).
- **Weekend:** Binary indicator (1 = weekend, 0 = weekday).

These features are available in advance, making them suitable for forecasting without introducing data leakage.

The target variable remains **Demand (GWh)**.

In [4]:
# Create calendar-based features

df["Hour"] = df["Period start"].dt.hour

df["Day_of_week"] = df["Period start"].dt.dayofweek

df["Month"] = df["Period start"].dt.month

df["Weekend"] = (df["Day_of_week"] >= 5).astype(int)

# Display the new features
display(
    df[
        ["Period start", "Region", "Demand (GWh)",
         "Hour", "Day_of_week", "Month", "Weekend"]
    ].head(10)
)

/var/folders/vm/wpy4jwz9593d3_cf75gx3y180000gn/T/ipykernel_1765/3921673257.py:5: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["Day_of_week"] = df["Period start"].dt.dayofweek


,Period start,Region,Demand (GWh),Hour,Day_of_week,Month,Weekend
0,2024-11-01 00:00:00,Upper North Island,0.541,0,4,11,0
1,2024-11-01 00:00:00,Central North Island,0.377,0,4,11,0
2,2024-11-01 00:00:00,Lower North Island,0.246,0,4,11,0
3,2024-11-01 00:00:00,Upper South Island,0.275,0,4,11,0
4,2024-11-01 00:00:00,Lower South Island,0.433,0,4,11,0
5,2024-11-01 00:30:00,Upper North Island,0.518,0,4,11,0
6,2024-11-01 00:30:00,Central North Island,0.370,0,4,11,0
7,2024-11-01 00:30:00,Lower North Island,0.234,0,4,11,0
8,2024-11-01 00:30:00,Upper South Island,0.264,0,4,11,0
9,2024-11-01 00:30:00,Lower South Island,0.425,0,4,11,0


### 3.2 Half-Hour Trading Period Feature

Electricity demand is recorded at 30-minute intervals, resulting in 48 standard trading periods per day.

The existing `Hour` feature identifies the hour but does not distinguish between the first and second half of that hour.

A new feature, `Trading_period`, is created to represent each half-hour interval.

For example:

- 00:00 = Period 0
- 00:30 = Period 1
- 01:00 = Period 2
- 12:00 = Period 24
- 23:30 = Period 47

This provides the model with more precise information about daily electricity consumption patterns.

Daylight saving transitions will be considered separately because some days contain fewer or more than 48 trading periods.

In [5]:
# Create half-hour trading period feature
df["Trading_period"] = (
    df["Period start"].dt.hour * 2
    + df["Period start"].dt.minute // 30
)

# Check the new feature
display(
    df[
        ["Period start", "Region", "Hour",
         "Trading_period", "Demand (GWh)"]
    ].head(15)
)

print("Minimum trading period:", df["Trading_period"].min())
print("Maximum trading period:", df["Trading_period"].max())

,Period start,Region,Hour,Trading_period,Demand (GWh)
0,2024-11-01 00:00:00,Upper North Island,0,0,0.541
1,2024-11-01 00:00:00,Central North Island,0,0,0.377
2,2024-11-01 00:00:00,Lower North Island,0,0,0.246
3,2024-11-01 00:00:00,Upper South Island,0,0,0.275
4,2024-11-01 00:00:00,Lower South Island,0,0,0.433
5,2024-11-01 00:30:00,Upper North Island,0,1,0.518
6,2024-11-01 00:30:00,Central North Island,0,1,0.370
7,2024-11-01 00:30:00,Lower North Island,0,1,0.234
8,2024-11-01 00:30:00,Upper South Island,0,1,0.264
9,2024-11-01 00:30:00,Lower South Island,0,1,0.425


Minimum trading period: 0
Maximum trading period: 47


### 3.3 Time Series Validation

Before generating historical demand features, the dataset must be checked for temporal consistency.

Electricity demand is recorded at 30-minute intervals across five regions.

The validation will investigate:

- Duplicate timestamps within each region.
- Gaps between consecutive observations.
- The number of observations recorded per region.
- Potential irregularities associated with daylight saving time.

These checks are necessary because historical demand features must refer to the correct previous observations.

Simply shifting rows without validating the time sequence could introduce incorrect lag values.

In [6]:
# Sort observations chronologically within each region
df = df.sort_values(
    ["Region ID", "Period start"]
).reset_index(drop=True)

# Check duplicate timestamps within regions
duplicates = df.duplicated(
    subset=["Region ID", "Period start"]
).sum()

# Calculate time differences between consecutive observations
df["Time_difference"] = (
    df.groupby("Region ID")["Period start"].diff()
)

# Identify intervals that are not exactly 30 minutes
irregular_intervals = df[
    df["Time_difference"].notna()
    & (df["Time_difference"] != pd.Timedelta(minutes=30))
]

print("Duplicate region-timestamp combinations:", duplicates)
print("Irregular time intervals:", len(irregular_intervals))

print("\nObservations per region:")
print(df.groupby("Region ID").size())

print("\nExamples of irregular intervals:")
display(
    irregular_intervals[
        ["Region ID", "Period start", "Time_difference"]
    ].head(15)
)

Duplicate region-timestamp combinations: 0
Irregular time intervals: 125

Observations per region:
Region ID
CNI    34999
LNI    34993
LSI    34974
UNI    34987
USI    34983
dtype: int64

Examples of irregular intervals:


,Region ID,Period start,Time_difference
485,CNI,2024-11-11 05:00:00,0 days 03:00:00
720,CNI,2024-11-16 04:00:00,0 days 02:00:00
735,CNI,2024-11-16 12:30:00,0 days 01:30:00
907,CNI,2024-11-20 04:30:00,0 days 02:30:00
2009,CNI,2024-12-13 04:00:00,0 days 01:00:00
2060,CNI,2024-12-14 06:00:00,0 days 01:00:00
2101,CNI,2024-12-15 03:00:00,0 days 01:00:00
2388,CNI,2024-12-21 03:00:00,0 days 01:00:00
2449,CNI,2024-12-22 10:00:00,0 days 01:00:00
2482,CNI,2024-12-23 05:00:00,0 days 03:00:00


### 3.4 Investigation of Irregular Time Intervals

Time series validation identified 125 intervals that differ from the expected 30-minute frequency.

Although no duplicate region-timestamp combinations were detected, several gaps and unusual timestamp intervals were observed.

Potential explanations include:

- Missing observations removed during data cleaning.
- Daylight saving time transitions.
- Special electricity market trading periods.

Before creating lag features, these irregularities must be investigated.

This is important because a lag of one observation does not necessarily correspond to 30 minutes when the time series contains gaps.

In [7]:
# Examine the distribution of irregular time intervals

interval_summary = (
    irregular_intervals["Time_difference"]
    .value_counts()
    .sort_index()
)

print("Distribution of irregular time intervals:")
display(interval_summary)

# Examine observations around daylight saving
dst_examples = df[
    (df["Period start"] >= "2025-04-06 01:00:00") &
    (df["Period start"] <= "2025-04-06 04:00:00") &
    (df["Region ID"] == "CNI")
]

print("\nObservations around 6 April 2025:")
display(
    dst_examples[
        ["Period start", "Period end",
         "Time_difference", "Demand (GWh)"]
    ]
)

Distribution of irregular time intervals:


Time_difference
0 days 00:10:00    30
0 days 01:00:00    34
0 days 01:30:00    24
0 days 02:00:00     6
0 days 02:30:00     2
0 days 03:00:00    18
0 days 03:30:00     1
0 days 04:00:00     5
0 days 05:00:00     5
Name: count, dtype: int64


Observations around 6 April 2025:


,Period start,Period end,Time_difference,Demand (GWh)
7449,2025-04-06 01:00:00,2025-04-06 01:30:00,0 days 00:30:00,0.347
7450,2025-04-06 01:30:00,2025-04-06 02:00:00,0 days 00:30:00,0.341
7451,2025-04-06 02:00:00,2025-04-06 02:30:00,0 days 00:30:00,0.337
7452,2025-04-06 02:30:00,2025-04-06 03:00:00,0 days 00:30:00,0.335
7453,2025-04-06 02:40:00,2025-04-06 03:10:00,0 days 00:10:00,0.334
7454,2025-04-06 02:50:00,2025-04-06 03:20:00,0 days 00:10:00,0.330
7455,2025-04-06 03:00:00,2025-04-06 03:30:00,0 days 00:10:00,0.329
7456,2025-04-06 03:30:00,2025-04-06 04:00:00,0 days 00:30:00,0.329
7457,2025-04-06 04:00:00,2025-04-06 04:30:00,0 days 00:30:00,0.331


### 3.5 Time Series Validation Summary

The time series validation identified 125 irregular intervals across the five electricity regions.

Most irregularities are consistent with gaps created when 264 observations containing missing demand values were removed during data cleaning.

Additional irregularities were identified around New Zealand's daylight-saving transition on 6 April 2025, including timestamps at 02:40 and 02:50.

These observations were retained because they represent special trading-period timestamps rather than necessarily indicating data errors.

No duplicate region-timestamp combinations were identified.

**Decision:** Retain all 174,936 valid observations and account for irregular time intervals when constructing historical demand features.

Historical demand features will use timestamp-based matching to avoid incorrectly treating the previous available observation as the previous 30-minute period.

## 4. Historical Demand Features

Historical electricity demand is expected to be an important predictor of future demand.

Electricity consumption generally changes gradually between consecutive trading periods.

Lag features allow the model to capture these short-term relationships.

### 4.1 Previous 30-Minute Demand

The first lag feature represents electricity demand exactly 30 minutes before the current observation.

Because the dataset contains missing observations and irregular timestamps, lag values will be matched using timestamps rather than simply shifting rows.

Missing lag values will remain missing when no observation exists at the required historical timestamp.

In [8]:
# Create a timestamp representing 30 minutes earlier
df["Previous_timestamp"] = (
    df["Period start"] - pd.Timedelta(minutes=30)
)

# Create a lookup table of historical demand
historical_demand = df[
    ["Region ID", "Period start", "Demand (GWh)"]
].copy()

historical_demand = historical_demand.rename(
    columns={
        "Period start": "Previous_timestamp",
        "Demand (GWh)": "Demand_lag_30min"
    }
)

# Match each observation with demand exactly 30 minutes earlier
df = df.merge(
    historical_demand,
    on=["Region ID", "Previous_timestamp"],
    how="left",
    validate="many_to_one"
)

# Display results
display(
    df[
        [
            "Period start",
            "Region",
            "Demand (GWh)",
            "Demand_lag_30min"
        ]
    ].head(12)
)

print(
    "Missing 30-minute lag values:",
    df["Demand_lag_30min"].isna().sum()
)

,Period start,Region,Demand (GWh),Demand_lag_30min
0,2024-11-01 00:00:00,Central North Island,0.377,NaN
1,2024-11-01 00:30:00,Central North Island,0.370,0.377
2,2024-11-01 01:00:00,Central North Island,0.361,0.370
3,2024-11-01 01:30:00,Central North Island,0.356,0.361
4,2024-11-01 02:00:00,Central North Island,0.351,0.356
5,2024-11-01 02:30:00,Central North Island,0.347,0.351
6,2024-11-01 03:00:00,Central North Island,0.341,0.347
7,2024-11-01 03:30:00,Central North Island,0.340,0.341
8,2024-11-01 04:00:00,Central North Island,0.345,0.340
9,2024-11-01 04:30:00,Central North Island,0.355,0.345


Missing 30-minute lag values: 120


In [9]:
print("Total observations:", len(df))

print(
    "Missing 30-minute lag values:",
    df["Demand_lag_30min"].isna().sum()
)

print(
    "Percentage missing:",
    round(df["Demand_lag_30min"].isna().mean() * 100, 3),
    "%"
)

Total observations: 174936
Missing 30-minute lag values: 120
Percentage missing: 0.069 %


### 4.2 Additional Historical Demand Features

Electricity demand is influenced by short-term consumption patterns and recurring daily and weekly cycles.

To capture these relationships, three additional lag features will be created:

- **Demand_lag_1h:** Electricity demand one hour earlier.
- **Demand_lag_24h:** Electricity demand at the same local time on the previous day.
- **Demand_lag_7d:** Electricity demand at the same local time one week earlier.

These features allow the forecasting model to learn different temporal relationships.

For example, electricity demand at 8:00 AM today may be similar to demand at 8:00 AM yesterday or at 8:00 AM on the same weekday last week.

Because the dataset contains missing observations and irregular trading periods, historical values will be matched using timestamps and regions rather than row positions.

Missing historical observations will remain as NaN values.

**Note:** The 24-hour and 7-day features use local clock timestamps. Daylight-saving transitions may affect their interpretation.

In [10]:
# Define the historical demand features
lag_features = {
    "Demand_lag_1h": pd.Timedelta(hours=1),
    "Demand_lag_24h": pd.Timedelta(days=1),
    "Demand_lag_7d": pd.Timedelta(days=7)
}

# Create a historical demand lookup table
historical_demand = df[
    ["Region ID", "Period start", "Demand (GWh)"]
].copy()

# Create each lag feature
for feature_name, time_delta in lag_features.items():

    lookup = historical_demand.rename(
        columns={
            "Period start": "Historical_timestamp",
            "Demand (GWh)": feature_name
        }
    )

    df["Historical_timestamp"] = (
        df["Period start"] - time_delta
    )

    df = df.merge(
        lookup,
        on=["Region ID", "Historical_timestamp"],
        how="left",
        validate="many_to_one"
    )

# Remove temporary column
df = df.drop(columns=["Historical_timestamp"])

# Display results
display(
    df[
        [
            "Period start",
            "Region",
            "Demand (GWh)",
            "Demand_lag_30min",
            "Demand_lag_1h",
            "Demand_lag_24h",
            "Demand_lag_7d"
        ]
    ].tail(10)
)

# Check missing values
print("\nMissing values in lag features:")

print(
    df[
        [
            "Demand_lag_30min",
            "Demand_lag_1h",
            "Demand_lag_24h",
            "Demand_lag_7d"
        ]
    ].isna().sum()
)

,Period start,Region,Demand (GWh),Demand_lag_30min,Demand_lag_1h,Demand_lag_24h,Demand_lag_7d
174926,2026-10-31 19:00:00,Upper South Island,0.355,0.356,0.356,0.431,0.371
174927,2026-10-31 19:30:00,Upper South Island,0.360,0.355,0.356,0.433,0.365
174928,2026-10-31 20:00:00,Upper South Island,0.369,0.360,0.355,0.432,0.366
174929,2026-10-31 20:30:00,Upper South Island,0.377,0.369,0.360,0.431,0.365
174930,2026-10-31 21:00:00,Upper South Island,0.383,0.377,0.369,0.435,0.362
174931,2026-10-31 21:30:00,Upper South Island,0.377,0.383,0.377,0.426,0.351
174932,2026-10-31 22:00:00,Upper South Island,0.371,0.377,0.383,0.413,0.344
174933,2026-10-31 22:30:00,Upper South Island,0.362,0.371,0.377,0.394,0.334
174934,2026-10-31 23:00:00,Upper South Island,0.361,0.362,0.371,0.388,0.334
174935,2026-10-31 23:30:00,Upper South Island,0.352,0.361,0.362,0.374,0.326



Missing values in lag features:
Demand_lag_30min     120
Demand_lag_1h        186
Demand_lag_24h       503
Demand_lag_7d       1957
dtype: int64


### 4.3 Historical Demand Feature Validation

Four historical demand features were successfully created:

- 30-minute lag
- 1-hour lag
- 24-hour lag
- 7-day lag

Missing-value analysis showed that all four features have relatively low proportions of missing observations.

The seven-day lag contains the highest proportion of missing values (1.119%), primarily because the first seven days have no available historical observations.

Additional missing values may result from gaps in the original dataset and irregular trading periods.

The lag features were created using timestamp-based matching within each electricity region, preventing historical demand values from being incorrectly assigned across regions.

**Decision:** Retain the missing values temporarily. Before model training, observations with unavailable required lag features will be excluded from the modelling dataset.

This approach avoids introducing artificial demand values through imputation.

In [11]:
# Define the historical demand features
lag_columns = [
    "Demand_lag_30min",
    "Demand_lag_1h",
    "Demand_lag_24h",
    "Demand_lag_7d"
]

# Count observations with all lag features available
complete_rows = df[lag_columns].notna().all(axis=1)

print("Total observations:", len(df))
print("Complete observations:", complete_rows.sum())
print("Observations with missing lag features:", (~complete_rows).sum())

print(
    "Percentage available for modelling:",
    round(complete_rows.mean() * 100, 2),
    "%"
)

Total observations: 174936
Complete observations: 172597
Observations with missing lag features: 2339
Percentage available for modelling: 98.66 %


## 5. Preparing the Modelling Dataset

Following feature engineering, 172,597 observations (98.66%) contain complete historical demand features.

The remaining 2,339 observations have at least one missing lag feature.

These missing values arise from unavailable historical observations, gaps in the original dataset, and irregular trading periods.

For the initial forecasting models, observations with incomplete lag features will be excluded.

The resulting modelling dataset will contain:

- Calendar-based features describing when electricity demand occurs.
- Regional information identifying the electricity consumption area.
- Historical demand features capturing short-term, daily, and weekly consumption patterns.
- The target variable, Demand (GWh).

The dataset will retain chronological timestamps so that training and testing can be separated by time rather than randomly.

This is important because forecasting models must be evaluated using future observations that were not available during training.

In [12]:
# Create a modelling dataset containing complete lag features
model_df = df.dropna(subset=lag_columns).copy()

# Select relevant columns
model_columns = [
    "Period start",
    "Region ID",
    "Region",
    "Demand (GWh)",
    "Hour",
    "Day_of_week",
    "Month",
    "Weekend",
    "Trading_period",
    "Demand_lag_30min",
    "Demand_lag_1h",
    "Demand_lag_24h",
    "Demand_lag_7d"
]

model_df = model_df[model_columns]

# Sort chronologically
model_df = model_df.sort_values(
    ["Period start", "Region ID"]
).reset_index(drop=True)

print("Modelling dataset shape:", model_df.shape)

print("\nMissing values:")
print(model_df.isna().sum())

print("\nDate range:")
print("Start:", model_df["Period start"].min())
print("End:", model_df["Period start"].max())

display(model_df.head())

Modelling dataset shape: (172597, 13)

Missing values:
Period start        0
Region ID           0
Region              0
Demand (GWh)        0
Hour                0
Day_of_week         0
Month               0
Weekend             0
Trading_period      0
Demand_lag_30min    0
Demand_lag_1h       0
Demand_lag_24h      0
Demand_lag_7d       0
dtype: int64

Date range:
Start: 2024-11-08 00:00:00
End: 2026-10-31 23:30:00


,Period start,Region ID,Region,Demand (GWh),Hour,Day_of_week,Month,Weekend,Trading_period,Demand_lag_30min,Demand_lag_1h,Demand_lag_24h,Demand_lag_7d
0,2024-11-08,CNI,Central North Island,0.370,0,4,11,0,0,0.383,0.403,0.368,0.377
1,2024-11-08,LNI,Lower North Island,0.236,0,4,11,0,0,0.251,0.269,0.244,0.246
2,2024-11-08,LSI,Lower South Island,0.424,0,4,11,0,0,0.429,0.454,0.432,0.433
3,2024-11-08,UNI,Upper North Island,0.534,0,4,11,0,0,0.564,0.600,0.526,0.541
4,2024-11-08,USI,Upper South Island,0.332,0,4,11,0,0,0.354,0.364,0.329,0.275


## 6. Exporting the Feature-Engineered Dataset

The feature engineering process produced a modelling dataset containing 172,597 observations and 13 columns.

The dataset includes:

- Calendar features: Hour, Day_of_week, Month, Weekend and Trading_period.
- Regional identifiers: Region ID and Region.
- Historical demand features: 30-minute, 1-hour, 24-hour and 7-day lags.
- Target variable: Demand (GWh).

All observations contain complete feature values.

The dataset covers 8 November 2024 to 31 October 2026.

The final dataset will be exported as a CSV file for use in the machine learning notebook.

The original cleaned dataset will be preserved separately.

### Next Stage: Model Development

The next notebook will focus on developing and evaluating electricity demand forecasting models.

Models will be trained and tested using chronological data splits to avoid training on future observations.

Model performance will be evaluated using MAE and RMSE.

In [13]:
# Save the feature-engineered dataset

output_path = (
    Path.cwd().parent
    / "data"
    / "electricity_demand_features.csv"
)

model_df.to_csv(output_path, index=False)

print("Feature-engineered dataset saved successfully.")
print("File:", output_path)
print("Observations:", len(model_df))
print("Columns:", len(model_df.columns))

Feature-engineered dataset saved successfully.
File: /Users/javiermorande/Desktop/nz-electricity-forecasting/data/electricity_demand_features.csv
Observations: 172597
Columns: 13
